# Support Ops Env — GRPO Training on Colab

**OpenEnv Hackathon Submission** | [HF Space](https://huggingface.co/spaces/raj23211/support-ops-env) | [GitHub](https://github.com/raj23211/support-ops-env)

Train a SaaS support-ops triage agent using **GRPO** with HF TRL. The agent learns to investigate cases across six apps (Inbox, CRM, Billing, Access, Policy, Comms), apply the right routing, consolidate duplicates, and draft a grounded reply — importing all training logic directly from the `support_ops_env` package.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://huggingface.co/spaces/raj23211/support-ops-env) (deterministic grader) |
| Training   | This Colab notebook (T4 by default, A100/L4 for vLLM fast path) |
| Model      | `Qwen/Qwen3-4B-Instruct-2507` + LoRA (4-bit NF4 on T4; bf16 + vLLM on A100) |
| Framework  | HF TRL GRPO (no vLLM on T4 — not enough VRAM for colocate) |

Why this model? Qwen3-4B-Instruct-2507 is dense 4B, **non-thinking** (clean JSON for our action parser), and known-stable with TRL GRPO. Qwen3.5-4B has open vLLM + QLoRA issues at the time of writing, so we keep it as a future upgrade.

The training signal is generated **at rollout time** by the agent interacting with the environment — there is no pre-collected dataset.

## 1. Install Dependencies

Installs the `support_ops_env` package (env client, models, tasks, training utils) with the `[train]` extra (TRL + vLLM + LoRA).

If `pip` shows **`Temporary failure in name resolution`** or **`No matching distribution found for trl … (from versions: none)`**, the runtime has **no working outbound internet/DNS**. Fix the VM network, then run the **network check** cell below before retrying.


In [ ]:
# Network check — run before pip if installs fail with DNS errors.
import socket
import urllib.request
for host in ("pypi.org", "files.pythonhosted.org", "huggingface.co"):
    socket.gethostbyname(host)
    print("resolve OK:", host)
_ = urllib.request.urlopen("https://pypi.org/simple/pip/", timeout=20).read(64)
print("HTTPS to PyPI OK — proceed to install.")


In [ ]:
# Qwen3-4B-Instruct-2507 requires transformers >= 5.0.
# On T4 we use 4-bit NF4 via bitsandbytes, and skip vLLM (doesn't fit alongside a 4B colocate KV cache).
!pip install -q --no-cache-dir \
    "transformers>=5.0" \
    "trl>=0.29.0" \
    "peft>=0.13.0" \
    "accelerate>=0.30" \
    "bitsandbytes>=0.43.0" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy"

# Install our env package (client + models + tasks + training utils). --no-deps
# avoids pip re-resolving torch / transformers we just installed.
!pip install -q --no-cache-dir --no-deps \
    "openenv-support-ops @ git+https://huggingface.co/spaces/raj23211/support-ops-env@main"
!pip install -q --no-cache-dir "openenv-core[core]>=0.2.3"

import transformers, trl, peft
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")

## 1b. (Optional) Install Unsloth

Only run this cell if you set `USE_UNSLOTH = True` in Section 2. Unsloth gives ~1.5× faster training and ~50% less VRAM on Qwen3.5-4B (official docs). Skip otherwise — the stable path above is enough.

In [ ]:
# Only needed when USE_UNSLOTH=True. Safe to re-run (idempotent).
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
import unsloth
print(f"unsloth : {unsloth.__version__}")

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets (key icon in the sidebar).

In [ ]:
import os

# Try Colab Secrets first; fall back to env var so this cell works anywhere.
try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not set. Either:")
        print("  - Click the key icon in the Colab sidebar, add HF_TOKEN, enable notebook access, re-run this cell.")
        print("  - Or run: os.environ['HF_TOKEN'] = 'hf_...'")

ENV_URL         = "https://raj23211-support-ops-env.hf.space"
USE_UNSLOTH     = False       # Flip True for Unsloth Qwen3.5-4B (bf16 LoRA, ~10 GB, ~1.5x faster)

if USE_UNSLOTH:
    MODEL_ID     = "unsloth/Qwen3.5-4B"
    LOAD_IN_4BIT = False      # Unsloth docs: QLoRA not recommended for Qwen3.5
    USE_VLLM     = False      # Unsloth uses its own generation (fast_inference=False)
    HUB_REPO     = "raj23211/support-ops-agent-qwen35-4b-unsloth"
else:
    MODEL_ID     = "Qwen/Qwen3-4B-Instruct-2507"
    LOAD_IN_4BIT = True       # NF4 + bf16 compute fits T4
    USE_VLLM     = False      # True only on A100/L4 24GB+
    HUB_REPO     = "raj23211/support-ops-agent-qwen3-4b-instruct"

NUM_EPISODES    = 50
NUM_GENERATIONS = 4           # per prompt — T4 handles 4
MAX_TURNS       = 12          # tighter horizon speeds up rollouts
DIFFICULTY      = "easy"      # curriculum: easy | medium | hard | expert | all

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Generations : {NUM_GENERATIONS}")
print(f"vLLM        : {USE_VLLM}")
print(f"4-bit NF4   : {LOAD_IN_4BIT}")

## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment (picks a task), and execute one investigative tool call to confirm round-trip works.

In [ ]:
from support_ops_env import SupportOpsAction, SupportOpsEnv, TASK_IDS

print(f"Connecting to {ENV_URL} ...")

env = SupportOpsEnv(base_url=ENV_URL).sync()

try:
    reset_result = env.reset(task_id=TASK_IDS[0])
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Task       : {obs.task_title}")
    print(f"Objective  : {obs.objective[:200]}...\n")

    step = env.step(
        SupportOpsAction(
            assistant_message="Listing inbox to find the primary case.",
            tool_calls=[{"name": "inbox.list_cases", "args": {}}],
            answer=None,
        )
    )
    print(f"--- smoke test step (reward={step.reward:.2f}) ---")
    for tr in step.observation.tool_results[-1:]:
        print(f"{tr.name} ok={tr.ok}\n{str(tr.result)[:400]}")

finally:
    if hasattr(env, 'close'):
        env.close()

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities from Package

All training logic (system prompt, rollout function, reward functions, helpers) is imported directly from `support_ops_env.train` — the same code used for H100 training. No duplication.

In [ ]:
import csv
import logging
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

from support_ops_env import get_training_utils

tu = get_training_utils()
SYSTEM_PROMPT         = tu["SYSTEM_PROMPT"]
rollout_once          = tu["rollout_once"]
reward_total          = tu["reward_total"]
reward_fields         = tu["reward_fields"]
reward_reply          = tu["reward_reply"]
reward_grounding      = tu["reward_grounding"]
plot_rewards          = tu["plot_rewards"]
patch_trl_vllm_compat = tu["patch_trl_vllm_compat"]

patch_trl_vllm_compat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("System prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])
print("...")
print("\nImported: rollout_once, reward_total, reward_fields, reward_reply, reward_grounding.")

## 5. GRPO Training Setup

Configure tokenizer, environment connection, reward logging, and the GRPO trainer — all using functions imported from the package.

In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    loaded_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=4096,
        load_in_4bit=False,
        load_in_16bit=True,
        full_finetuning=False,
        fast_inference=False,   # required for Qwen3.5 GRPO (Unsloth docs)
    )
    loaded_model = FastLanguageModel.get_peft_model(
        loaded_model,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        max_seq_length=4096,
    )
else:
    loaded_model = None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

env = SupportOpsEnv(base_url=ENV_URL).sync()
dataset = Dataset.from_dict(
    {"prompt": ["Triage and resolve this support operations case."] * NUM_EPISODES}
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(f"outputs/support-ops-grpo-{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_totals = []

with open(reward_log_path, "w", newline="") as fh:
    csv.writer(fh).writerow([
        "episode", "total_reward", "field_reward", "reply_reward", "grounding_reward", "timestamp",
    ])

def log_episode(total_r, field_r, reply_r, ground_r):
    episode_counter[0] += 1
    all_totals.append(total_r)
    with open(reward_log_path, "a", newline="") as fh:
        csv.writer(fh).writerow([
            episode_counter[0], total_r, field_r, reply_r, ground_r, datetime.now().isoformat(),
        ])
    last10 = all_totals[-10:]
    logger.info(
        f"Episode {episode_counter[0]}: reward={total_r:+.2f} | "
        f"mean={sum(all_totals)/len(all_totals):+.2f}, mean(10)={sum(last10)/len(last10):+.2f}"
    )

from support_ops_env import get_curriculum_task_ids

curriculum = get_curriculum_task_ids(DIFFICULTY)
print(f"Curriculum [{DIFFICULTY}] -> {curriculum}")
_task_cursor = [0]

def _next_task_id():
    tid = curriculum[_task_cursor[0] % len(curriculum)]
    _task_cursor[0] += 1
    return tid

def rollout_func(prompts, trainer):
    results = {k: [] for k in [
        "prompt_ids", "completion_ids", "logprobs",
        "total_reward", "field_reward", "reply_reward", "grounding_reward",
    ]}
    for _ in prompts:
        ep = rollout_once(trainer, env, tokenizer, SYSTEM_PROMPT, MAX_TURNS,
                          task_id=_next_task_id())
        for k in results:
            results[k].append(ep[k])
        log_episode(
            ep["total_reward"], ep["field_reward"], ep["reply_reward"], ep["grounding_reward"],
        )
    return results

print(f"Training setup complete. Output: {output_dir}")

In [ ]:
import torch

grpo_kwargs = dict(
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    temperature=1.0,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=3,
)

if USE_VLLM and not USE_UNSLOTH:
    grpo_kwargs.update(
        use_vllm=True,
        vllm_mode="colocate",
        vllm_gpu_memory_utilization=0.5,
    )

if LOAD_IN_4BIT and not USE_UNSLOTH:
    from transformers import BitsAndBytesConfig
    # NF4 weights + bf16 compute. LoRA adapters stay in bf16 (QLoRA).
    grpo_kwargs["model_init_kwargs"] = {
        "torch_dtype": torch.bfloat16,
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        ),
    }

grpo_config = GRPOConfig(**grpo_kwargs)

if USE_UNSLOTH:
    # Unsloth already attached LoRA via get_peft_model above.
    model_arg = loaded_model
    peft_config = None
else:
    from peft import LoraConfig
    model_arg = MODEL_ID
    peft_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

trainer = GRPOTrainer(
    model=model_arg,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_fields, reward_reply, reward_grounding],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print("GRPOTrainer initialized")
print(f"  Unsloth       : {USE_UNSLOTH}")
print(f"  vLLM          : {USE_VLLM}")
print(f"  4-bit NF4     : {LOAD_IN_4BIT}")
print(f"  VRAM now      : {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 6. Train!

Launch GRPO training. Each episode: `reset` → agent investigates via tool calls → agent drafts a reply and submits → deterministic grader scores → GRPO gradient update.

In [ ]:
print("Starting GRPO training...")
print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {ENV_URL}")
print()

try:
    trainer.train()
finally:
    if hasattr(env, "close"):
        env.close()

trainer.save_model(str(output_dir))
print(f"\nModel saved to {output_dir}")

## 7. Reward Curves

Visualize training progress using `plot_rewards` from the package.

In [ ]:
plot_rewards(reward_log_path, output_dir / "reward_curve.png")

import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(str(output_dir / "reward_curve.png"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

print(f"Reward curve saved to {output_dir / 'reward_curve.png'}")

## 8. Push to Hub (Optional)

Upload the trained LoRA adapter to Hugging Face Hub.

In [ ]:
# Uncomment to push:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")